## <font color="royalblue">**Extracción de datos**</font>
### <font color="royalblue">**Web Scraping de Indeed**</font>
En esta archivo se definen todos los pasos para extraer del portal web Indeed ofertas de empleo en Data Analyst para Barcelona.
Un proceso de web scraping involucra acceder a páginas web que poseen medios de protección, por tal motivo el ingreso se realizo con 2 
herramientas; el uso de selenium.webdriver y logeo con cuenta a través de cookies.
- Se cargan las librerias necesarias para el proceso (Selenium, BeautifulSoap, requests).

**Paso 1. Acceso al portal web**
- Las funciones *guardar_cookies*, *iniciar_sesion_con_cookies*; nos 
permiten almacenar las cookies de una cuenta e iniciar sesión con la 
misma.   


**Paso 2. Extraccción de datos** 
- Las funciones *ir_a_busqueda*, *scroll_infinito_linkedin*, 
*extraer_id*, *extraer_jocards*, *scrapear_paginas*: permiten ir a 
LinkedIn jobs, cargar las ofertas de empleo e iniciar la extraccion de 
datos.  
- Se crea la función *guardar_cookies*: ingresa el navegador a la web LinkedIn, manualmente se inicia sesión introduciendo email y contraseña,
para poder almacenarla en la variable *cookies* y exportarla como archivo .json. SE EJECUTA SOLO UNA VEZ.

**Paso 2. Extraccción de datos** 
- Las funciones *ir_a_busqueda*, 
*scroll_infinito_linkedin*, 

**Paso 3. Extracción de descripción de oferta**
- Las funciones *obtener_descripcion_indeed*, *extraer_descripciones*: en el caso de las 
descripciones que implica acceder a cada id y url de forma individual se hizo en un proceso con Scrape.do y BeautifulSoap.


- Finalmente se creo un dataframe *df_Indeed* para visualizar los datos y fueron exportados en archivo .csv para posterior preprocesamiento.

In [ ]:
import undetected_chromedriver as uc
from selenium.webdriver.common.by import By
from selenium.common.exceptions import NoSuchElementException
from selenium.webdriver.common.action_chains import ActionChains
import requests
from bs4 import BeautifulSoup
import os
import time
import random
import requests
import json
import pandas as pd


In [ ]:
# Función para iniciar el navegador
def iniciar_chrome():
        
    # Ruta del Chrome normal (estable)
    chrome_path = r"C:\Program Files\Google\Chrome\Application\chrome.exe"

    # Carpeta donde guardaremos un perfil REAL de Chrome
    user_data_dir = r"C:\ChromeProfileReal"
    profile_dir = "Profile1"

    # Crear carpeta si no existe
    os.makedirs(user_data_dir, exist_ok=True)

    options = uc.ChromeOptions()
    options.binary_location = chrome_path

    # Usar un perfil real (muy importante para Indeed)
    options.add_argument(f"--user-data-dir={user_data_dir}")
    options.add_argument(f"--profile-directory={profile_dir}")

    # User-agent realista (coincide con tu Chrome 145)
    options.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/145.0.0.0 Safari/537.36"
    )

    # Anti detección
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument("--no-first-run")
    options.add_argument("--no-default-browser-check")
    options.add_argument("--homepage=about:blank")
    options.add_argument("--disable-features=ChromeWhatsNewUI")
    options.add_argument("--window-size=1366,768")
    options.add_argument("--start-maximized")

    # Lanzar Chrome normal con undetected_chromedriver
    driver = uc.Chrome(
        options=options,
        version_main=145  # Coincide con tu Chrome real
    )

    # Ocultar navigator.webdriver
    driver.execute_cdp_cmd(
        "Page.addScriptToEvaluateOnNewDocument",
        {
            "source": """
                Object.defineProperty(navigator, 'webdriver', {
                    get: () => undefined
                });
            """
        }
    )

    driver.get("about:blank")
    return driver


In [ ]:
# Función para hacer scroll
def scroll_humano(driver):

    for _ in range(8):
        driver.execute_script(
            f"window.scrollBy(0, {random.randint(200, 500)});"
        )
        time.sleep(random.uniform(0.4, 1.2))


In [ ]:
# Función para esperar a que se carguen los JobCards
def esperar_jobcards(driver):

    for i in range(30):  # 30 segundos de espera total
        cards = driver.find_elements(By.CSS_SELECTOR, '[data-testid="jobCard"]')
        if len(cards) > 0:
            print(f"JobCards detectados: {len(cards)}")
            return True

        # Scroll suave para activar el renderizado
        driver.execute_script("window.scrollBy(0, 300);")
        time.sleep(1)

    return False


In [ ]:
# Función para mover el mouse
def mover_mouse(driver):

    try:
        # Mover el mouse al buscador de Indeed (siempre visible)
        search_box = driver.find_element(By.CSS_SELECTOR, 'input[name="q"]')
        ActionChains(driver).move_to_element(search_box).perform()
        time.sleep(0.5)
    except:
        pass


In [ ]:
# Función para cerrar popups
def cerrar_popups(driver):
    # Popup de iniciar sesión
    try:
        btn_close = driver.find_element(By.CSS_SELECTOR, 'button[aria-label="Close"]')
        btn_close.click()
        print("Popup de login cerrado.")
    except:
        pass

    # Popup de restauración de Chrome
    try:
        btn_restore = driver.find_element(By.CSS_SELECTOR, 'button[aria-label="Restore"]')
        btn_restore.click()
        print("Popup de restauración cerrado.")
    except:
        pass

    # Popup de extensiones (Adobe, etc.)
    try:
        btn_remove = driver.find_element(By.XPATH, "//button[contains(text(),'Remove')]")
        btn_remove.click()
        print("Popup de extensión cerrado.")
    except:
        pass


In [ ]:
# Función para aceptar términos
def aceptar_terminos(driver):
    try:
        btn_accept = driver.find_element(By.XPATH, "//button[contains(text(), 'Accept Terms')]")
        btn_accept.click()
        print("Términos aceptados.")
        time.sleep(2)
    except:
        pass


In [ ]:
# Función para extraer URLs
def extraer_urls(driver):
        
    time.sleep(2)

    # Scroll inicial para activar lazy loading
    driver.execute_script("window.scrollTo(0, 400);")
    time.sleep(1)

    urls = set()
    base_url = "https://es.indeed.com/jobs?"
    # 1. Selector principal: a[data-jk]
    enlaces = driver.find_elements(By.CSS_SELECTOR, "a[data-jk]")
    for a in enlaces:
        jk = a.get_attribute("data-jk")
        if jk:
            url = f"{base_url}jk={jk}"
            urls.add(url)

    # 2. Selector alternativo: div.job_seen_beacon
    cards = driver.find_elements(By.CSS_SELECTOR, "div.job_seen_beacon")
    for card in cards:
        try:
            a = card.find_element(By.CSS_SELECTOR, "a[data-jk]")
            jk = a.get_attribute("data-jk")
            if jk:
                urls.add(f"{base_url}jk={jk}")
        except:
            pass

    print(f"Detectadas {len(urls)} ofertas")
    return list(urls)


ahora primero se inciara la sesion no se aplica este codigo

In [ ]:
# Función para scrapear Indeed
def scrapear_indeed(driver, query="data+analyst", location="Barcelona+provincia", paginas=3):
    todas = []
    base_url = "https://es.indeed.com/jobs?"
    for pagina in range(paginas):
        start = pagina * 10
        url = f"{base_url}q={query}&l={location}&start={start}"
        print("Navegando a:", url)

        driver.get(url)
        time.sleep(3)

        cerrar_popups(driver)
        time.sleep(1)

        urls = extraer_urls(driver)
        print(f"Página {pagina+1}: {len(urls)} URLs encontradas")

        todas.extend(urls)

    return todas
# usa cerrar_popups, extraer_urls

Nuevo flujo de login (robusto y compatible con captcha)

In [ ]:
# Función para esperar el captcha
def esperar_captcha(driver):
    print("Completa la verificación humana en Indeed...")
    print("Selenium esperará hasta que desaparezca el captcha.")

    for _ in range(120):  # hasta 2 minutos
        try:
            # Si aparece el botón 'Sign in', significa que ya pasaste el captcha
            driver.find_element(By.LINK_TEXT, "Sign in")
            print("Captcha superado.")
            return True
        except:
            time.sleep(2)

    print("No se detectó que el captcha fuera completado.")
    return False


#### Función para iniciar sesión Indeed

In [ ]:
# Función para iniciar sesión
def iniciar_sesion_indeed(driver, email):
    base_url = "https://es.indeed.com/jobs?"
    driver.get(base_url)
    time.sleep(3)

    # Botón "Iniciar sesión" arriba a la derecha
    try:
        btn_signin = driver.find_element(By.LINK_TEXT, "Iniciar sesión")
        btn_signin.click()
        time.sleep(3)
    except:
        print("No se encontró el botón 'Iniciar sesión'.")
        return False

    # Campo de correo (tu versión usa name="__email")
    try:
        campo_email = driver.find_element(By.CSS_SELECTOR, 'input[name="__email"]')
        campo_email.send_keys(email)
        time.sleep(1)

        btn_continue = driver.find_element(By.CSS_SELECTOR, 'button[type="submit"]')
        btn_continue.click()
        time.sleep(3)
    except Exception as e:
        print("No se pudo introducir el correo:", e)
        return False

    print("Introduce el código MFA enviado a tu correo Outlook.")
    print("Selenium esperará hasta que completes el login manualmente...")

    # Esperar a que la sesión esté iniciada
    for _ in range(120):  # hasta 2 minutos
        try:
            driver.find_element(By.CSS_SELECTOR, '[data-testid="user-menu-button"]')
            print("Sesión iniciada correctamente.")
            return True
        except:
            time.sleep(2)

    print("No se detectó inicio de sesión.")
    return False


#### Guardar cookies después del login

In [ ]:
# Función para guardar las cookies
def guardar_cookies(driver, archivo="cookies_indeed.json"):
    cookies = driver.get_cookies()
    with open(archivo, "w") as f:
        json.dump(cookies, f)
    print("Cookies guardadas.")


#### Cargar cookies en ejecuciones futuras  

In [ ]:
# Función para cargar las cookies
def cargar_cookies(driver, archivo="cookies_indeed.json"):
    import json
    import time
    base_url = "https://es.indeed.com/jobs?"
    driver.get(base_url)
    time.sleep(3)

    with open(archivo, "r") as f:
        cookies = json.load(f)

    for cookie in cookies:
        driver.add_cookie(cookie)

    driver.get(base_url)
    time.sleep(3)
    print("Cookies cargadas, sesión restaurada.")


#### Flujo completo de primera ejecución (con MFA)

In [ ]:
driver = iniciar_chrome()

if iniciar_sesion_indeed(driver, "castellanorn@outlook.com"):
    guardar_cookies(driver)

driver.quit()
# usa guardar_cookies

modal de terminos

#### Flujo para ejecuciones futuras (sin MFA)

In [ ]:
driver = iniciar_chrome()
cargar_cookies(driver)
aceptar_terminos(driver)

# Ya puedes scrapear sin login wall
urls = scrapear_indeed(driver, query="data+analyst",location="Barcelona,Barcelona+provincia", paginas=1) #l=Barcelona%2C+Barcelona+provincia

driver.quit()


Cada oferta

In [ ]:
# Función para extraer el detalle de una oferta
def extraer_detalle(driver, url):

    datos = {
        "url": url,
        "titulo": None,
        "empresa": None,
        "ubicacion": None,
        "salario": None,
        "tipo": None,
        "descripcion": None
    }

    try:
        driver.get(url)
        time.sleep(2)

        # Título
        try:
            datos["titulo"] = driver.find_element(By.CSS_SELECTOR, 'h1[data-testid="jobTitle"]').text
        except:
            pass

        # Empresa
        try:
            datos["empresa"] = driver.find_element(By.CSS_SELECTOR, 'div[data-testid="companyInfo"] span').text
        except:
            pass

        # Ubicación
        try:
            datos["ubicacion"] = driver.find_element(By.CSS_SELECTOR, 'div[data-testid="jobLocation"]').text
        except:
            pass

        # Salario (si aparece)
        try:
            datos["salario"] = driver.find_element(By.CSS_SELECTOR, 'div[data-testid="attribute_snippet_testid"]').text
        except:
            pass

        # Descripción
        try:
            datos["descripcion"] = driver.find_element(By.CSS_SELECTOR, '#jobDescriptionText').text
        except:
            pass

    except Exception as e:
        print(f"Error extrayendo {url}: {e}")

    return datos


In [ ]:
# Función para extraer detalles de ofertas con url
def extraer_detalles_de_urls(driver, urls):
    resultados = []

    for i, url in enumerate(urls, start=1):
        print(f"Extrayendo oferta {i}/{len(urls)}")
        datos = extraer_detalle(driver, url)
        resultados.append(datos)

    return resultados
# usa extraer_detalle

In [ ]:
# Función para extraer descripción
def agregar_descripciones(driver, ofertas):
        
    for i, oferta in enumerate(ofertas, start=1):
        url = oferta["url"]
        print(f"Extrayendo descripción {i}/{len(ofertas)}")

        driver.get(url)
        time.sleep(2)

        try:
            descripcion = driver.find_element(By.CSS_SELECTOR, "#jobDescriptionText").text
        except:
            descripcion = None

        oferta["descripcion"] = descripcion

    return ofertas


In [ ]:
def extraer_descripcion(driver, url):
    
    driver.get(url)
    time.sleep(2)

    try:
        return driver.find_element(By.CSS_SELECTOR, "#jobDescriptionText").text
    except:
        return None


In [ ]:
def extraer_jobcards(driver):
    time.sleep(2)

    ofertas = []

    cards = driver.find_elements(By.CSS_SELECTOR, "div.job_seen_beacon")

    for card in cards:
        datos = {
            "id": None,
            "titulo": None,
            "empresa": None,
            "ubicacion": None,
            "salario": None,
            "tipo_contrato": None,
            "modalidad": None,
            "url": None
        }

        # Título
        try:
            a = card.find_element(By.CSS_SELECTOR, "h2 a")
            datos["titulo"] = a.text
        except:
            pass

        # ID y URL
        try:
            jk = a.get_attribute("data-jk")
            datos["id"] = jk
            datos["url"] = f"https://es.indeed.com/viewjob?jk={jk}"
        except:
            pass

        # Empresa
        try:
            datos["empresa"] = card.find_element(By.CSS_SELECTOR, "span[data-testid='company-name']").text
        except:
            pass

        # Ubicación
        try:
            datos["ubicacion"] = card.find_element(By.CSS_SELECTOR, "div[data-testid='text-location']").text
        except:
            pass

        # Salario, tipo de contrato y modalidad
        try:
            cont = card.find_element(By.CSS_SELECTOR, "div.jobMetaDataGroup")
            spans = cont.find_elements(By.TAG_NAME, "span")
            textos = [s.text.strip() for s in spans]

            for t in textos:
                t_low = t.lower()

                # Salario
                if "€" in t or "eur" in t_low:
                    datos["salario"] = t

                # Tipo de contrato
                elif any(p in t_low for p in ["jornada", "tiempo", "indefinido", "temporal", "contrato"]):
                    datos["tipo_contrato"] = t

                # Modalidad
                elif any(p in t_low for p in ["remoto", "híbrido", "hibrido", "presencial"]):
                    datos["modalidad"] = t

        except:
            pass

        ofertas.append(datos)

    print(f"Extraídas {len(ofertas)} ofertas desde jobCards")
    return ofertas


In [ ]:
driver = iniciar_chrome()
cargar_cookies(driver)
aceptar_terminos(driver)

base_url = "https://es.indeed.com/jobs?"
driver.get(f"{base_url}q=data+analyst&l=Barcelona%2C+Barcelona+provincia")

ofer = extraer_jobcards(driver)

for o in ofer:
    print(o)


In [ ]:
driver = iniciar_chrome()
cargar_cookies(driver)
aceptar_terminos(driver)

urls = scrapear_indeed(driver, query="data+analyst", location="Barcelona", paginas=1)

detalles = extraer_detalles_de_urls(driver, urls)

driver.quit()

for d in detalles:
    print("\n-------------------------------")
    for k, v in d.items():
        print(f"{k}: {v}")


In [ ]:
# Función para scrapear todas las páginas
def scrapear_indeed_paginas(driver, query, location, paginas=5):

    todas = []
    ids_vistos = set()
    base_url = "https://es.indeed.com/jobs?"
    for pagina in range(paginas):
        start = pagina * 10
        url = f"{base_url}q={query}&l={location}&start={start}"
        print(f"Navegando a página {pagina+1}: {url}")

        driver.get(url)
        time.sleep(2)

        # Scroll para activar lazy loading
        driver.execute_script("window.scrollTo(0, 400);")
        time.sleep(1)

        ofertas = extraer_jobcards(driver)

        # Evitar duplicados
        nuevas = []
        for o in ofertas:
            if o["id"] not in ids_vistos:
                ids_vistos.add(o["id"])
                nuevas.append(o)

        print(f"Página {pagina+1}: {len(nuevas)} ofertas nuevas")

        todas.extend(nuevas)

        # Si no hay ofertas nuevas, paramos
        if len(nuevas) == 0:
            print("No hay más resultados, fin del scraping.")
            break

    print(f"\nTotal acumulado: {len(todas)} ofertas")
    return todas
# usa extraer_jobcards

In [ ]:
driver = iniciar_chrome()
cargar_cookies(driver)
aceptar_terminos(driver)

# 1. Scrapear jobCards de todas las páginas
ofertas = scrapear_indeed_paginas(
    driver,
    query="data+analyst",
    location="Barcelona",
    paginas=2
)

# 2. Añadir descripción a cada oferta
ofertas = agregar_descripciones(driver, ofertas)

driver.quit()


Cookies cargadas, sesión restaurada.
Navegando a página 1: https://es.indeed.com/jobs?q=data+analyst&l=Barcelona&start=0
Extraídas 0 ofertas desde jobCards
Página 1: 0 ofertas nuevas
No hay más resultados, fin del scraping.

Total acumulado: 0 ofertas


In [61]:
df_Indeed=pd.DataFrame(ofertas)
df_Indeed

,id,titulo,empresa,ubicacion,salario,tipo_contrato,modalidad,url,descripcion
0,ef8da29b4067d8ce,FP&A Analyst,Freightos,"Trabajo híbrido in 08018 Barcelona, Barcelona ...",None,Jornada completa,None,https://es.indeed.com/viewjob?jk=ef8da29b4067d8ce,None
1,d01ca385b03a8ab5,BI Analyst,Grupo Planeta,"Barcelona, Barcelona provincia",None,None,None,https://es.indeed.com/viewjob?jk=d01ca385b03a8ab5,None
2,cdef0123456789ab,,,,None,None,None,https://es.indeed.com/viewjob?jk=cdef0123456789ab,None
3,8c331ce1dbb312a4,Junior Data Analyst,Holded,"Trabajo híbrido in 08039 Barcelona, Barcelona ...",None,Jornada completa,None,https://es.indeed.com/viewjob?jk=8c331ce1dbb312a4,None
4,e84341c140a96071,Junior Data Analyst,EXOGROUP,"Trabajo híbrido in 08005 Barcelona, Barcelona ...",None,Jornada completa,None,https://es.indeed.com/viewjob?jk=e84341c140a96071,None
...,...,...,...,...,...,...,...,...,...
134,06e2a78e8dc73bd0,OutSystems Developer Consultant,Zurich Insurance,"Barcelona, Barcelona provincia",None,Jornada completa,None,https://es.indeed.com/viewjob?jk=06e2a78e8dc73bd0,We Are Waiting for You\n\nHi there!\nI am Álva...
135,7e0cce10992dd6d1,Consultor Inmobiliario Industrial (Barcelona),Engel & Völkers España,"Barcelona, Barcelona provincia",None,Jornada completa,None,https://es.indeed.com/viewjob?jk=7e0cce10992dd6d1,Descripción:\nEngel & Völkers es una empresa l...
136,508aa6b0a7ffa408,Senior II Back-End Engineer,Preply,"Trabajo híbrido in Barcelona, Barcelona provincia",None,Jornada completa,None,https://es.indeed.com/viewjob?jk=508aa6b0a7ffa408,"We power people’s progress.\nAt Preply, we’re ..."
137,6e71d97dea2fd3a6,Senior Product Manager - Growth,Wallapop,"Trabajo híbrido in Barcelona, Barcelona provincia",None,None,None,https://es.indeed.com/viewjob?jk=6e71d97dea2fd3a6,Wallapop is a Barcelona based scale-up driven ...


In [ ]:
df_Indeed.to_csv('df_Indeed.csv')

In [ ]:
df_Indeed_inic=pd.read_csv('df_Indeed.csv')
df_Indeed_inic



,Unnamed: 0,id,titulo,empresa,ubicacion,salario,tipo_contrato,modalidad,url,descripcion
0,0,ef8da29b4067d8ce,FP&A Analyst,Freightos,"Trabajo híbrido in 08018 Barcelona, Barcelona ...",NaN,Jornada completa,NaN,https://es.indeed.com/viewjob?jk=ef8da29b4067d8ce,NaN
1,1,d01ca385b03a8ab5,BI Analyst,Grupo Planeta,"Barcelona, Barcelona provincia",NaN,NaN,NaN,https://es.indeed.com/viewjob?jk=d01ca385b03a8ab5,NaN
2,2,cdef0123456789ab,NaN,NaN,NaN,NaN,NaN,NaN,https://es.indeed.com/viewjob?jk=cdef0123456789ab,NaN
3,3,8c331ce1dbb312a4,Junior Data Analyst,Holded,"Trabajo híbrido in 08039 Barcelona, Barcelona ...",NaN,Jornada completa,NaN,https://es.indeed.com/viewjob?jk=8c331ce1dbb312a4,NaN
4,4,e84341c140a96071,Junior Data Analyst,EXOGROUP,"Trabajo híbrido in 08005 Barcelona, Barcelona ...",NaN,Jornada completa,NaN,https://es.indeed.com/viewjob?jk=e84341c140a96071,NaN
...,...,...,...,...,...,...,...,...,...,...
134,134,06e2a78e8dc73bd0,OutSystems Developer Consultant,Zurich Insurance,"Barcelona, Barcelona provincia",NaN,Jornada completa,NaN,https://es.indeed.com/viewjob?jk=06e2a78e8dc73bd0,We Are Waiting for You\n\nHi there!\nI am Álva...
135,135,7e0cce10992dd6d1,Consultor Inmobiliario Industrial (Barcelona),Engel & Völkers España,"Barcelona, Barcelona provincia",NaN,Jornada completa,NaN,https://es.indeed.com/viewjob?jk=7e0cce10992dd6d1,Descripción:\nEngel & Völkers es una empresa l...
136,136,508aa6b0a7ffa408,Senior II Back-End Engineer,Preply,"Trabajo híbrido in Barcelona, Barcelona provincia",NaN,Jornada completa,NaN,https://es.indeed.com/viewjob?jk=508aa6b0a7ffa408,"We power people’s progress.\nAt Preply, we’re ..."
137,137,6e71d97dea2fd3a6,Senior Product Manager - Growth,Wallapop,"Trabajo híbrido in Barcelona, Barcelona provincia",NaN,NaN,NaN,https://es.indeed.com/viewjob?jk=6e71d97dea2fd3a6,Wallapop is a Barcelona based scale-up driven ...


In [ ]:
# Funciones para extraer descripción de cada oferta con scrape.do
API_KEY = "caf1b1a17b8842248c9c4c845da388f283cec0b156d"

def obtener_descripcion_indeed(url):
    endpoint = f"https://api.scrape.do?token={API_KEY}&url={url}"
    r = requests.get(endpoint)
    soup = BeautifulSoup(r.text, "html.parser")
    desc = soup.select_one("div#jobDescriptionText")
    return desc.get_text("\n", strip=True) if desc else None

def extraer_descripciones(df):
    resultados = []

    for i, row in df.iterrows():
        print(f"{i+1}/{len(df)} → {row['id']}")
        descripcion = obtener_descripcion_indeed(row["url"])

        resultados.append({
            "id": row["id"],
            "url": row["url"],
            "descripcion": descripcion
        })

    return pd.DataFrame(resultados)
# usa obtener_descripcion_indeed

In [ ]:
# Extraer descripción de cada oferta y union con resto de datos
df_desc = extraer_descripciones(df_Indeed_inic)

df_final = df_Indeed_inic.merge(df_desc, on="id", how="left")

df_final


1/139 → ef8da29b4067d8ce
2/139 → d01ca385b03a8ab5
3/139 → cdef0123456789ab
4/139 → 8c331ce1dbb312a4
5/139 → e84341c140a96071
6/139 → a897d014547833a7
7/139 → 44e2f2db3bee962f
8/139 → 0a543d40dea506b1
9/139 → 3a577d8e5f1dc4bf
10/139 → 4496d6a782288478
11/139 → 588301838e968e55
12/139 → 89ef7e9d17dc9cc3
13/139 → a90082a231e49905
14/139 → deaba371a923219d
15/139 → 79a593cf3f137459
16/139 → 1f3607849d3ad2d4
17/139 → 3f05a49cc7108aeb
18/139 → d1825536b86d14f1
19/139 → 3fb5ce6e1489e3c9
20/139 → 475af0816afd1e8d
21/139 → 6208ff914489c529
22/139 → 73709ef1d59351e3
23/139 → 789abcdef0123456
24/139 → ff6cb25c43468584
25/139 → ae5d560ff2a96373
26/139 → d241f9555bb3fe0c
27/139 → b6f1626b2c6e7a3e
28/139 → 5de868a449bf6bb6
29/139 → ee071c6c0b5e83cd
30/139 → feb7da58e39a229e
31/139 → e2623ca10983bac9
32/139 → 2eccb57a2d3b176b
33/139 → 2ef6b22156fbf1ee
34/139 → caf2c57f65ba0e10
35/139 → bece160541b196dd
36/139 → d2948d61d903914f
37/139 → f1e2d3c4b5a67890
38/139 → abab105be3483a6c
39/139 → 859ff93cf658

,Unnamed: 0,id,titulo,empresa,ubicacion,salario,tipo_contrato,modalidad,url_x,descripcion_x,url_y,descripcion_y
0,0,ef8da29b4067d8ce,FP&A Analyst,Freightos,"Trabajo híbrido in 08018 Barcelona, Barcelona ...",NaN,Jornada completa,NaN,https://es.indeed.com/viewjob?jk=ef8da29b4067d8ce,NaN,https://es.indeed.com/viewjob?jk=ef8da29b4067d8ce,About Us\nAlmost every single thing that you e...
1,1,d01ca385b03a8ab5,BI Analyst,Grupo Planeta,"Barcelona, Barcelona provincia",NaN,NaN,NaN,https://es.indeed.com/viewjob?jk=d01ca385b03a8ab5,NaN,https://es.indeed.com/viewjob?jk=d01ca385b03a8ab5,Activa | Planeta Innovación Somos el proveedor...
2,2,cdef0123456789ab,NaN,NaN,NaN,NaN,NaN,NaN,https://es.indeed.com/viewjob?jk=cdef0123456789ab,NaN,https://es.indeed.com/viewjob?jk=cdef0123456789ab,None
3,3,8c331ce1dbb312a4,Junior Data Analyst,Holded,"Trabajo híbrido in 08039 Barcelona, Barcelona ...",NaN,Jornada completa,NaN,https://es.indeed.com/viewjob?jk=8c331ce1dbb312a4,NaN,https://es.indeed.com/viewjob?jk=8c331ce1dbb312a4,"Join the team. Make an impact.\nAt\nHolded\n, ..."
4,4,e84341c140a96071,Junior Data Analyst,EXOGROUP,"Trabajo híbrido in 08005 Barcelona, Barcelona ...",NaN,Jornada completa,NaN,https://es.indeed.com/viewjob?jk=e84341c140a96071,NaN,https://es.indeed.com/viewjob?jk=e84341c140a96071,About ExoClick:\nExoClick is an innovative and...
...,...,...,...,...,...,...,...,...,...,...,...,...
134,134,06e2a78e8dc73bd0,OutSystems Developer Consultant,Zurich Insurance,"Barcelona, Barcelona provincia",NaN,Jornada completa,NaN,https://es.indeed.com/viewjob?jk=06e2a78e8dc73bd0,We Are Waiting for You\n\nHi there!\nI am Álva...,https://es.indeed.com/viewjob?jk=06e2a78e8dc73bd0,None
135,135,7e0cce10992dd6d1,Consultor Inmobiliario Industrial (Barcelona),Engel & Völkers España,"Barcelona, Barcelona provincia",NaN,Jornada completa,NaN,https://es.indeed.com/viewjob?jk=7e0cce10992dd6d1,Descripción:\nEngel & Völkers es una empresa l...,https://es.indeed.com/viewjob?jk=7e0cce10992dd6d1,None
136,136,508aa6b0a7ffa408,Senior II Back-End Engineer,Preply,"Trabajo híbrido in Barcelona, Barcelona provincia",NaN,Jornada completa,NaN,https://es.indeed.com/viewjob?jk=508aa6b0a7ffa408,"We power people’s progress.\nAt Preply, we’re ...",https://es.indeed.com/viewjob?jk=508aa6b0a7ffa408,None
137,137,6e71d97dea2fd3a6,Senior Product Manager - Growth,Wallapop,"Trabajo híbrido in Barcelona, Barcelona provincia",NaN,NaN,NaN,https://es.indeed.com/viewjob?jk=6e71d97dea2fd3a6,Wallapop is a Barcelona based scale-up driven ...,https://es.indeed.com/viewjob?jk=6e71d97dea2fd3a6,None


In [56]:
df_final.to_csv('df_full_Indeed.csv')

Proceso de revision de resultados de descripciones

In [ ]:
df_sin_desc = df_final[
    (df_final["descripcion_x"].isna() | (df_final["descripcion_x"] == "")) &
    (df_final["descripcion_y"].isna() | (df_final["descripcion_y"] == ""))
]

In [57]:
df_final["descripcion_final"] = df_final["descripcion_x"]

mask = df_final["descripcion_final"].isna() | (df_final["descripcion_final"] == "")
df_final.loc[mask, "descripcion_final"] = df_final.loc[mask, "descripcion_y"]


In [58]:
df_final

,Unnamed: 0,id,titulo,empresa,ubicacion,salario,tipo_contrato,modalidad,url_x,descripcion_x,url_y,descripcion_y,descripcion_final
0,0,ef8da29b4067d8ce,FP&A Analyst,Freightos,"Trabajo híbrido in 08018 Barcelona, Barcelona ...",NaN,Jornada completa,NaN,https://es.indeed.com/viewjob?jk=ef8da29b4067d8ce,NaN,https://es.indeed.com/viewjob?jk=ef8da29b4067d8ce,About Us\nAlmost every single thing that you e...,About Us\nAlmost every single thing that you e...
1,1,d01ca385b03a8ab5,BI Analyst,Grupo Planeta,"Barcelona, Barcelona provincia",NaN,NaN,NaN,https://es.indeed.com/viewjob?jk=d01ca385b03a8ab5,NaN,https://es.indeed.com/viewjob?jk=d01ca385b03a8ab5,Activa | Planeta Innovación Somos el proveedor...,Activa | Planeta Innovación Somos el proveedor...
2,2,cdef0123456789ab,NaN,NaN,NaN,NaN,NaN,NaN,https://es.indeed.com/viewjob?jk=cdef0123456789ab,NaN,https://es.indeed.com/viewjob?jk=cdef0123456789ab,None,None
3,3,8c331ce1dbb312a4,Junior Data Analyst,Holded,"Trabajo híbrido in 08039 Barcelona, Barcelona ...",NaN,Jornada completa,NaN,https://es.indeed.com/viewjob?jk=8c331ce1dbb312a4,NaN,https://es.indeed.com/viewjob?jk=8c331ce1dbb312a4,"Join the team. Make an impact.\nAt\nHolded\n, ...","Join the team. Make an impact.\nAt\nHolded\n, ..."
4,4,e84341c140a96071,Junior Data Analyst,EXOGROUP,"Trabajo híbrido in 08005 Barcelona, Barcelona ...",NaN,Jornada completa,NaN,https://es.indeed.com/viewjob?jk=e84341c140a96071,NaN,https://es.indeed.com/viewjob?jk=e84341c140a96071,About ExoClick:\nExoClick is an innovative and...,About ExoClick:\nExoClick is an innovative and...
...,...,...,...,...,...,...,...,...,...,...,...,...,...
134,134,06e2a78e8dc73bd0,OutSystems Developer Consultant,Zurich Insurance,"Barcelona, Barcelona provincia",NaN,Jornada completa,NaN,https://es.indeed.com/viewjob?jk=06e2a78e8dc73bd0,We Are Waiting for You\n\nHi there!\nI am Álva...,https://es.indeed.com/viewjob?jk=06e2a78e8dc73bd0,None,We Are Waiting for You\n\nHi there!\nI am Álva...
135,135,7e0cce10992dd6d1,Consultor Inmobiliario Industrial (Barcelona),Engel & Völkers España,"Barcelona, Barcelona provincia",NaN,Jornada completa,NaN,https://es.indeed.com/viewjob?jk=7e0cce10992dd6d1,Descripción:\nEngel & Völkers es una empresa l...,https://es.indeed.com/viewjob?jk=7e0cce10992dd6d1,None,Descripción:\nEngel & Völkers es una empresa l...
136,136,508aa6b0a7ffa408,Senior II Back-End Engineer,Preply,"Trabajo híbrido in Barcelona, Barcelona provincia",NaN,Jornada completa,NaN,https://es.indeed.com/viewjob?jk=508aa6b0a7ffa408,"We power people’s progress.\nAt Preply, we’re ...",https://es.indeed.com/viewjob?jk=508aa6b0a7ffa408,None,"We power people’s progress.\nAt Preply, we’re ..."
137,137,6e71d97dea2fd3a6,Senior Product Manager - Growth,Wallapop,"Trabajo híbrido in Barcelona, Barcelona provincia",NaN,NaN,NaN,https://es.indeed.com/viewjob?jk=6e71d97dea2fd3a6,Wallapop is a Barcelona based scale-up driven ...,https://es.indeed.com/viewjob?jk=6e71d97dea2fd3a6,None,Wallapop is a Barcelona based scale-up driven ...


In [ ]:
df_final.drop(columns=["Unnamed: 0", "descripcion_x", "descripcion_y", "url_x"], inplace=True)
df_final.rename(columns={"url_y": "url"}, inplace=True)
df_final.rename(columns={"descripcion_final": "descripcion"}, inplace=True)



In [ ]:
urls_faltantes = df_final[["id", "url"]].copy()
df_Indeed = df_final.copy()

In [ ]:
# Guarda el DataFrame en un archivo CSV
df_Indeed.to_csv('df_Indeed.csv')